# Sonic v5: six-target sound classifier

This notebook trains the event classifier on exactly six target labels: `door_knock`, `coughing`, `baby_crying`, `dog_bark`, `cat_meow`, and `footsteps`. A separate internal `background` reject class uses non-target clips so the app can tell when none of those six events is detected; it is not a displayed event label.

The exported classifier keeps the existing 20-slot Flutter output contract, with probabilities populated only for the six trained targets and zeros for the other slots. Model quality must be judged from held-out recordings, per-class metrics, and (when supplied) a separate phone-recorded test manifest. Do not claim a target accuracy before running the notebook.

The attached environment currently has ESC-50 only. ESC-50 provides 40 recordings per target class, which is too little to establish robust real-world performance by itself. Add more independent target recordings and the held-out phone test set before treating the reported result as deployment-ready.


## Added sources and split discipline

The collector also supports VocalSound, CatMeows, InfantCry-DBL, and your own phone development manifest. VocalSound is grouped by speaker; CatMeows by cat; InfantCry-DBL is training-only. See docs/model-training.md for source licenses, expected layout, and phone-manifest formats. Keep raw audio outside Git.



In [ ]:
%pip install -q librosa soundfile pandas numpy scikit-learn matplotlib tensorflow

## 1. Config

In [ ]:
import os, glob, json, time, zlib, hashlib, shutil, subprocess, re
import numpy as np, pandas as pd

SEED = 42
np.random.seed(SEED)
SMOKE = os.environ.get('SONIC_SMOKE') == '1'          # tiny run to test the pipeline end to end

# ---------- dataset roots (raw archives/audio stay outside Git) ----------
# Run from the repository root, or set SONIC_DATA_ROOT and SONIC_ESC50_PATH.
DATA_ROOT = os.path.abspath(os.environ.get('SONIC_DATA_ROOT', 'datasets'))
PATHS = {
    'esc': os.path.abspath(os.environ.get('SONIC_ESC50_PATH', os.path.join(DATA_ROOT, 'ESC-50'))),
    'fsd_manifest': os.path.join(DATA_ROOT, 'fsd50k_selected_manifest.csv'),
    'fsd_audio': os.path.join(DATA_ROOT, 'fsd50k_selected'),
    'urban': os.path.join(DATA_ROOT, 'UrbanSound8K'),
    'musan': os.path.join(DATA_ROOT, 'musan'),
    'extra': os.path.join(DATA_ROOT, 'extra_data'),
    'vocal_sound': os.path.join(DATA_ROOT, 'VocalSound', 'audio_16k'),
    'catmeows': os.path.join(DATA_ROOT, 'CatMeows'),
    'dil': os.path.join(DATA_ROOT, 'DIL-DCASE26-Dev'),
    'infantcry': os.path.join(DATA_ROOT, 'InfantCry-DBL'),
    'phone_dev_manifest': os.path.join(DATA_ROOT, 'phone_train_manifest.csv'),
    'field_manifest': os.path.join(DATA_ROOT, 'realworld_test_manifest.csv'),
}
USE = {'esc': True, 'fsd': True, 'urban': True, 'musan': True, 'extra': True,
       'vocal_sound': True, 'catmeows': True, 'dil': True, 'infantcry': True,
       'phone_dev': True, 'field': True}

# ---------- training knobs ----------
EPOCHS = int(os.environ.get('SONIC_EPOCHS', 2 if SMOKE else 60))   # early stopping usually ends sooner
BATCH, PATIENCE = 32, 12
GAIN_LOG10 = 1.5               # random gain in log10-power units: +-15 dB
NOISE_MIX_PROB = 0.65          # mix target events with held-out background TRAIN windows
MIXUP_PROB = 0.35              # conservative power-domain mixup

# ---------- data knobs ----------
SILENCE_RMS = 1e-3             # drop digital-silence windows everywhere
REL_LOUD = 0.20                # TRAIN, target classes only: drop windows < 20% of the clip's loudest window
MAX_SECONDS = 30               # load at most 30 s per file
MAX_WIN_TRAIN = 10             # windows per clip in train (loudest for targets, random for background)
MAX_WIN_EVAL = 6               # windows per clip in val/test (evenly spaced, NOT loudness-filtered)
CLIP_CAP = {'train': 600, 'val': 150, 'test': 150}   # max clips per (class, source, sub-class, split)
WIN_CAP_TRAIN_CLASS, WIN_CAP_TRAIN_BG = 2500, 6000
WIN_CAP_EVAL_BG = 4000
MUSAN_FILES_PER_GROUP = 20 if SMOKE else 300
MIN_TRAIN_CLIPS = 4 if SMOKE else 20  # require at least 20 distinct TRAIN recordings per target and background
MIN_EVENT_THRESHOLD = 0.70    # never lower the app's 70% event gate
FA_TARGET = 0.02               # validation false-alarm target on background windows

OPTIONAL_DOWNLOAD_MUSAN = False   # ~11 GB, 10-20 min on Colab. Needs ~25 GB of free disk.
if OPTIONAL_DOWNLOAD_MUSAN and not os.path.isdir(PATHS['musan']):
    subprocess.run(['wget', '-q', 'https://www.openslr.org/resources/17/musan.tar.gz'], check=True)
    subprocess.run(['tar', '-xzf', 'musan.tar.gz', 'musan/speech', 'musan/music', 'musan/noise/free-sound'], check=True)
    os.remove('musan.tar.gz')

## 2. App contract + features (must match the Dart side)

Order **must** match `lib/core/ml/sound_labels.dart`. The model also has an internal `background` class that
is **not** in the app's 20 slots: its probability is simply dropped, so on speech/room noise every slot stays low.

In [ ]:
import librosa

APP_CATEGORIES = [
    'smoke_alarm', 'fire_alarm', 'siren', 'baby_crying', 'glass_breaking',
    'doorbell', 'door_knock', 'dog_bark', 'phone_ringing', 'name_calling',
    'car_horn', 'appliance_beep', 'water_running', 'thunderstorm', 'alarm_clock',
    'cat_meow', 'footsteps', 'coughing', 'snoring', 'danger_explosion',
]   # DO NOT reorder
NUM_APP_CATEGORIES = len(APP_CATEGORIES)   # keep the 20-slot Flutter contract

# These are the ONLY displayed event classes trained by this notebook.
TARGET_CATEGORIES = [
    'door_knock', 'coughing', 'baby_crying', 'dog_bark', 'cat_meow', 'footsteps',
]

esc50_to_app = {
    'door_wood_knock': 'door_knock', 'coughing': 'coughing',
    'crying_baby': 'baby_crying', 'dog': 'dog_bark', 'cat': 'cat_meow',
    'footsteps': 'footsteps',
}
# ESC-50 classes held out of training COMPLETELY: used only to test "Teach a Sound" on sounds the model never saw.
ESC_NOVEL = ['keyboard_typing', 'mouse_click', 'clock_tick', 'can_opening']
# All non-target ESC-50 classes are useful hard negatives for this six-class task.
# ESC_NOVEL stays reserved for the notebook's separate prototype evaluation.
ESC_BG_EXCLUDE = set()

URBAN_TARGET = {'dog_bark': 'dog_bark'}
# Every UrbanSound8K class not in URBAN_TARGET is mapped to internal background.

SR, N_FFT, HOP, N_MELS, F_MIN, F_MAX = 16000, 512, 256, 64, 50.0, 8000.0
WIN = SR
EXPECTED_FRAMES = ((WIN - N_FFT) // HOP) + 1      # 61
HOP_S = SR // 2                                   # 0.5 s hop between windows

def extract_logmel(w):
    S = librosa.feature.melspectrogram(y=w, sr=SR, n_fft=N_FFT, hop_length=HOP, win_length=N_FFT, window='hann',
                                       n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX, power=2.0, center=False)
    return np.log10(np.maximum(S, 1e-6)).T.astype(np.float32)     # (61, 64), same as mel_spectrogram.dart

assert extract_logmel(np.zeros(WIN, dtype=np.float32)).shape == (EXPECTED_FRAMES, N_MELS)

AUDIO_EXT = ('.wav', '.mp3', '.ogg', '.flac', '.m4a')
def list_audio(folder):
    return sorted(p for p in glob.glob(os.path.join(folder, '**', '*'), recursive=True) if p.lower().endswith(AUDIO_EXT))

def load_wave(path, offset=0.0):
    wav, _ = librosa.load(path, sr=SR, mono=True, offset=offset, duration=MAX_SECONDS)
    return wav

def pick_windows(wav, split, is_bg, rng):
    # Split a waveform into 1 s windows (0.5 s hop) and choose which to keep.
    # train/target: loud windows only, loudest first. train/background: random (quiet room noise is wanted).
    # val/test: evenly spaced, no loudness filter, so the numbers are not flattering.
    if len(wav) < WIN:
        wav = np.pad(wav, (0, WIN - len(wav)))
    starts = np.arange(0, len(wav) - WIN + 1, HOP_S)
    ws = np.stack([wav[s:s + WIN] for s in starts])
    r = np.sqrt((ws ** 2).mean(axis=1))
    idx = np.where(r >= SILENCE_RMS)[0]
    if len(idx) == 0:
        return []
    if split == 'train':
        if is_bg:
            if len(idx) > MAX_WIN_TRAIN:
                idx = np.sort(rng.choice(idx, MAX_WIN_TRAIN, replace=False))
        else:
            idx = idx[r[idx] >= REL_LOUD * r[idx].max()]
            if len(idx) > MAX_WIN_TRAIN:
                idx = np.sort(idx[np.argsort(-r[idx])[:MAX_WIN_TRAIN]])
    elif len(idx) > MAX_WIN_EVAL:
        idx = idx[np.linspace(0, len(idx) - 1, MAX_WIN_EVAL).round().astype(int)]
    return [ws[i] for i in idx]

## 3. Collect clips from every dataset that is present

Nothing is loaded here, only a list of (file, class, source, split). Splits never mix a recording into two sets:
ESC-50 and UrbanSound8K use their **official folds**; FSD50K uses its own dev/eval flag; MUSAN/extra split by file.

VocalSound non-cough vocalizations are hard negatives for cough. CatMeows uses cat-grouped splits. InfantCry-DBL contributes training clips only. Phone development data can only enter train/validation, while the separate phone test manifest is never used for training.



In [ ]:
items, novel_items = [], []
field_test_non_target_labels = []
rng_items = np.random.default_rng(SEED)

def add(path, label, source, split, sub, rand_off=False):
    items.append(dict(path=path, label=label, source=source, split=split, sub=sub, rand_off=rand_off))

def hash_split(path):                       # deterministic 70/15/15 split by file
    h = zlib.crc32(path.encode()) % 100
    return 'train' if h < 70 else 'val' if h < 85 else 'test'

found = {}

# ---------------- ESC-50 ----------------
if USE['esc']:
    if not os.path.isdir(PATHS['esc']):
        os.makedirs(os.path.dirname(PATHS['esc']), exist_ok=True)
        subprocess.run(['git', 'clone', '-q', '--depth', '1',
                        'https://github.com/karolpiczak/ESC-50.git', PATHS['esc']], check=True)
    meta = pd.read_csv(os.path.join(PATHS['esc'], 'meta', 'esc50.csv'))
    if SMOKE:
        meta = meta.sample(frac=1, random_state=SEED).groupby('category').head(10)
    for r in meta.itertuples():
        path = os.path.join(PATHS['esc'], 'audio', r.filename)
        split = 'train' if r.fold in (1, 2, 3) else 'val' if r.fold == 4 else 'test'
        if r.category in esc50_to_app:
            add(path, esc50_to_app[r.category], 'esc50', split, r.category)
        elif r.category in ESC_NOVEL:
            novel_items.append((path, r.category))
        elif r.category not in ESC_BG_EXCLUDE:
            add(path, 'background', 'esc50', split, r.category)
    found['esc50'] = len(meta)

# ---------------- DIL-DCASE26 development set: held-out domain test clips ----------------
# Official dev-test files remain test-only. Dev-train is split by clip for train/validation.
# Its non-target labels (alarm, engine, fire, piano, speech, telephone_ringing) are hard negatives.
DIL_TO_APP = {'baby_cry': 'baby_crying', 'bark': 'dog_bark',
              'dog_bark': 'dog_bark', 'footsteps': 'footsteps',
              'knock': 'door_knock', 'knocking': 'door_knock'}
if USE.get('dil') and os.path.isdir(PATHS['dil']):
    n_dil = 0
    for domain in ('D2', 'D3'):
        base = os.path.join(PATHS['dil'], domain)
        metadata_dir = os.path.join(base, 'metadata')
        for split_name, split_dir in (('train', 'dev-train'), ('test', 'dev-test')):
            manifest = os.path.join(metadata_dir, f"{domain.lower()}-{split_dir}.csv")
            audio_dir = os.path.join(base, f"{domain.lower()}-{split_dir}")
            if not (os.path.isfile(manifest) and os.path.isdir(audio_dir)):
                continue
            dil_df = pd.read_csv(manifest)
            for _, r in dil_df.iterrows():
                filename = str(r['filename'])
                source_label = str(r['class']).lower()
                path = os.path.join(audio_dir, filename)
                if not os.path.isfile(path):
                    continue
                label = DIL_TO_APP.get(source_label, 'background')
                split = 'test' if split_name == 'test' else hash_split(f'dil:{domain}:{filename}')
                add(path, label, 'dil_dcase26', split, f'{domain}_{source_label}')
                n_dil += 1
    if n_dil:
        found['dil_dcase26'] = n_dil
    else:
        print('DIL-DCASE26 folder found but no matching D2/D3 metadata/audio -> skipped')
else:
    print('DIL-DCASE26 development audio not found -> skipped')

# ---------------- VocalSound: cough positives and vocal hard negatives ----------------
# Keep speakers grouped across splits; filenames look like f0003_0_cough.wav.
if USE.get('vocal_sound') and os.path.isdir(PATHS['vocal_sound']):
    n_vocal = 0
    for p in list_audio(PATHS['vocal_sound']):
        m = re.match(r'^([fmo]\d+)_\d+_(.+)$', os.path.splitext(os.path.basename(p))[0].lower())
        if not m:
            continue
        speaker, vocal_class = m.groups()
        if vocal_class not in {'cough', 'coughing', 'laughter', 'sigh', 'sneeze', 'sniff', 'throatclearing'}:
            continue
        label = 'coughing' if vocal_class in {'cough', 'coughing'} else 'background'
        add(p, label, 'vocalsound', hash_split('vocalsound:' + speaker), vocal_class)
        n_vocal += 1
    found['vocalsound'] = n_vocal
else:
    print('VocalSound 16 kHz audio not found -> skipped')

# ---------------- CatMeows: group by cat ID to avoid clip-level leakage ----------------
if USE.get('catmeows') and os.path.isdir(PATHS['catmeows']):
    n_cat = 0
    for p in list_audio(PATHS['catmeows']):
        stem = os.path.splitext(os.path.basename(p))[0]
        m = re.match(r'^[BFI]_([^_]+)_', stem, flags=re.IGNORECASE)
        if not m:
            continue
        cat_id = m.group(1)
        add(p, 'cat_meow', 'catmeows', hash_split('catmeows:' + cat_id), cat_id)
        n_cat += 1
    found['catmeows'] = n_cat
else:
    print('CatMeows audio not found -> skipped')

# ---------------- InfantCry-DBL: supplementary positives, training-only ----------------
# It is excluded from validation/test; use independent phone recordings for field claims.
if USE.get('infantcry') and os.path.isdir(PATHS['infantcry']):
    infant_files = list_audio(PATHS['infantcry'])
    for p in infant_files:
        lower_path = p.lower().replace('\\', '/')
        tier = 'tier1' if 'tier1' in lower_path or 'dunstan-core' in lower_path else 'tier2'
        add(p, 'baby_crying', 'infantcry_dbl', 'train', tier)
    found['infantcry_dbl_train_only'] = len(infant_files)
else:
    print('InfantCry-DBL audio not found -> skipped')

# ---------------- FSD50K (from your collection notebook's manifest) ----------------
if USE['fsd'] and os.path.exists(PATHS['fsd_manifest']) and os.path.isdir(PATHS['fsd_audio']):
    fm = pd.read_csv(PATHS['fsd_manifest'])
    fm = fm.copy()
    fm['app_category'] = fm['app_category'].astype(str).where(
        fm['app_category'].astype(str).isin(TARGET_CATEGORIES), 'background')
    fm['path'] = fm['fname'].apply(lambda f: os.path.join(PATHS['fsd_audio'], f'{f}.wav'))
    fm = fm[fm['path'].apply(os.path.exists)]
    for r in fm[fm['source_split'] == 'eval'].itertuples():
        add(r.path, r.app_category, 'fsd50k', 'test', r.app_category)
    for cls, g in fm[fm['source_split'] == 'dev'].groupby('app_category'):
        g = g.sample(frac=1.0, random_state=SEED)
        n_val = int(0.2 * len(g)) if len(g) >= 5 else 0          # clip-level 80/20 (no uploader info in the manifest)
        for i, r in enumerate(g.itertuples()):
            add(r.path, cls, 'fsd50k', 'val' if i < n_val else 'train', cls)
    found['fsd50k'] = len(fm)
else:
    print('FSD50K manifest/folder not found -> skipped')

# ---------------- UrbanSound8K (official folds) ----------------
um_csv = os.path.join(PATHS['urban'], 'metadata', 'UrbanSound8K.csv')
if USE['urban'] and os.path.exists(um_csv):
    um = pd.read_csv(um_csv).rename(columns={'class': 'cls'})
    keep = []
    for c in URBAN_TARGET:                                   # prefer foreground (salience 1) clips for target classes
        allc = um[um.cls == c]
        fg = allc[allc.salience == 1]
        keep.append(fg if len(fg) >= 60 else allc)
    keep.append(um[~um.cls.isin(URBAN_TARGET)])  # every non-target urban sound is background
    um = pd.concat(keep)
    for r in um.itertuples():
        path = os.path.join(PATHS['urban'], 'audio', f'fold{r.fold}', r.slice_file_name)
        split = 'train' if r.fold <= 7 else 'val' if r.fold <= 9 else 'test'
        if r.cls in URBAN_TARGET:
            add(path, URBAN_TARGET[r.cls], 'urbansound8k', split, r.cls)
        else:
            add(path, 'background', 'urbansound8k', split, r.cls)
    found['urbansound8k'] = len(um)
else:
    print('UrbanSound8K not found -> skipped')

# ---------------- MUSAN (speech + music + free-sound noise, all -> background) ----------------
# noise/sound-bible is deliberately NOT used: it contains beeps, dial tones and rings that sound like real alerts.
MUSAN_GROUPS = {'speech': 'speech', 'music': 'music', 'noise': os.path.join('noise', 'free-sound')}
if USE['musan'] and os.path.isdir(PATHS['musan']):
    n_musan = 0
    for name, rel in MUSAN_GROUPS.items():
        fs = list_audio(os.path.join(PATHS['musan'], rel))
        if not fs:
            print(f'  MUSAN: no audio found under {rel}'); continue
        pick = rng_items.choice(len(fs), min(MUSAN_FILES_PER_GROUP, len(fs)), replace=False)
        for i in pick:
            add(fs[i], 'background', 'musan', hash_split(fs[i]), name, rand_off=True)
        n_musan += len(pick)
    found['musan'] = n_musan
else:
    print('MUSAN not found -> skipped (background will come from ESC-50 / UrbanSound8K only)')

# ---------------- extra_data: AudioSet clips, Freesound, your own recordings ----------------
if USE['extra'] and os.path.isdir(PATHS['extra']):
    n_extra = 0
    for cat in sorted(os.listdir(PATHS['extra'])):
        fs = list_audio(os.path.join(PATHS['extra'], cat))
        if not fs:
            continue
        label = cat if cat in TARGET_CATEGORIES else 'background'
        for p in fs:
            add(p, label, 'extra', hash_split(p), cat)
        n_extra += len(fs)
    if n_extra:
        found['extra'] = n_extra

print('\nFiles found per source:', found or 'NOTHING')
df_items = pd.DataFrame(items)
assert len(df_items), 'No audio found at all. Check PATHS.'

# cap clips per (class, source, sub-class, split) so no single source or recording style dominates
parts = [g.sample(min(len(g), CLIP_CAP[split]), random_state=SEED)
         for (label, source, sub, split), g in df_items.groupby(['label', 'source', 'sub', 'split'])]
df_items = pd.concat(parts).sort_values(['source', 'label', 'sub', 'path']).reset_index(drop=True)
df_items['clip'] = np.arange(len(df_items))

# ---------------- Phone recordings for development: train/validation only ----------------
phone_dev_manifest = PATHS['phone_dev_manifest']
if USE.get('phone_dev') and os.path.isfile(phone_dev_manifest):
    pm = pd.read_csv(phone_dev_manifest)
    required = {'path', 'app_category', 'recording_id', 'split'}
    missing = required - set(pm.columns)
    if missing:
        raise ValueError(f'Phone development manifest missing columns: {sorted(missing)}')
    pm['split'] = pm['split'].astype(str).str.lower()
    if not set(pm['split']).issubset({'train', 'val'}):
        raise ValueError('phone_train_manifest.csv split must contain only train or val.')
    if (pm.groupby('recording_id')['split'].nunique() > 1).any():
        raise ValueError('A recording_id occurs in both train and val; keep each session in one split.')
    pm['app_category'] = pm['app_category'].astype(str).where(
        pm['app_category'].astype(str).isin(TARGET_CATEGORIES), 'background')
    def resolve_phone_path(p):
        p = str(p)
        return p if os.path.isabs(p) else os.path.join(os.path.dirname(os.path.abspath(phone_dev_manifest)), p)
    pm['path'] = pm['path'].map(resolve_phone_path)
    missing_files = pm.loc[~pm['path'].map(os.path.isfile), 'path'].tolist()
    if missing_files:
        raise FileNotFoundError(f'{len(missing_files)} phone development files are missing; first: {missing_files[0]}')
    for r in pm.itertuples():
        add(r.path, r.app_category, 'phone_dev', r.split, str(r.recording_id))
    found['phone_dev'] = len(pm)
else:
    print('No phone development manifest -> source-only validation; collect phone clips before field claims.')

# ---------------- Real-world recordings: test-only, recorded on intended phone/device ----------------
field_manifest = PATHS['field_manifest']
if USE.get('field', True) and os.path.isfile(field_manifest):
    fm = pd.read_csv(field_manifest)
    required = {'path', 'app_category', 'recording_id'}
    missing = required - set(fm.columns)
    if missing:
        raise ValueError(f"Real-world test manifest missing columns: {sorted(missing)}")
    field_test_non_target_labels = sorted(set(fm['app_category'].astype(str)) - set(TARGET_CATEGORIES) - {'background'})
    fm['app_category'] = fm['app_category'].astype(str).where(
        fm['app_category'].astype(str).isin(TARGET_CATEGORIES), 'background')
    if fm.empty:
        raise ValueError('Real-world manifest has no supported app_category labels.')
    def resolve_field_path(p):
        p = str(p)
        return p if os.path.isabs(p) else os.path.join(os.path.dirname(os.path.abspath(field_manifest)), p)
    fm['path'] = fm['path'].map(resolve_field_path)
    missing_files = fm.loc[~fm['path'].map(os.path.isfile), 'path'].tolist()
    if missing_files:
        raise FileNotFoundError(f"{len(missing_files)} real-world audio files are missing; first: {missing_files[0]}")
    for r in fm.itertuples():
        # All phone recordings in this manifest are test-only and never enter training/validation.
        add(r.path, r.app_category, 'realworld_phone', 'test', str(r.recording_id))
    found['realworld_phone_test'] = len(fm)
else:
    print('No real-world phone test manifest -> field performance is NOT measured in this run.')

print('\nCLIPS per class x source (after caps, all splits):')
print(pd.crosstab(df_items['label'], df_items['source'], margins=True).to_string())

## 4. Decide trained classes, build log-mel windows (cached), split

* The six requested target classes are mandatory and must each meet `MIN_TRAIN_CLIPS`; otherwise the notebook stops. Every other sound is mapped to the internal background/reject class or held out for the separate prototype check. Untrained app slots stay at 0.
* The feature cache key includes the file list and every window/feature setting, so it can't silently go stale.

In [ ]:
n_train_clips = df_items[df_items.split == 'train'].groupby('label').size()
TRAINED = list(TARGET_CATEGORIES)
missing_targets = {c: int(n_train_clips.get(c, 0)) for c in TARGET_CATEGORIES
                   if n_train_clips.get(c, 0) < MIN_TRAIN_CLIPS}
if missing_targets:
    raise ValueError(f'Cannot train the required six classes: too few distinct TRAIN clips: {missing_targets}. Add labeled clips; do not silently drop a class.')
assert n_train_clips.get('background', 0) >= MIN_TRAIN_CLIPS, 'Not enough internal background/reject clips.'
LABELS = TRAINED + ['background']
NUM_TRAINED, NUM_CLASSES = len(TRAINED), len(LABELS)
BG = NUM_TRAINED                                  # index of the background class (maps to NO app slot)
L2I = {c: i for i, c in enumerate(LABELS)}
APP_INDEX = [APP_CATEGORIES.index(c) for c in TRAINED]
NOT_TRAINED = [c for c in APP_CATEGORIES if c not in TRAINED]
print(f'Training exactly {NUM_TRAINED} target classes + one internal background/reject class')
print('Target class order:', TRAINED)
print('Other app slots kept zero:', NOT_TRAINED)
field_labels = set(df_items.loc[df_items.source == 'realworld_phone', 'label']) if 'realworld_phone' in set(df_items.source) else set()
FIELD_TEST_UNTRAINED = sorted(field_labels - set(LABELS))
if FIELD_TEST_UNTRAINED:
    print('WARNING: unexpected real-world labels remain outside the six targets/background:', FIELD_TEST_UNTRAINED)

df_items = df_items[df_items.label.isin(LABELS)].reset_index(drop=True)

def build_features(df):
    X, rows = [], []
    for n, r in enumerate(df.itertuples()):
        prng = np.random.default_rng(zlib.crc32(r.path.encode()))
        try:
            off = 0.0
            if r.rand_off:                                        # long MUSAN files: random 30 s chunk, stable per file
                off = float(prng.uniform(0, max(0.0, librosa.get_duration(path=r.path) - MAX_SECONDS)))
            wav = load_wave(r.path, off)
        except Exception as e:
            print('  skipping', r.path, '->', repr(e)[:90]); continue
        for w in pick_windows(wav, r.split, r.label == 'background', prng):
            X.append(extract_logmel(w))
            rows.append((L2I[r.label], r.split, r.source, r.sub, r.clip))
        if (n + 1) % 500 == 0:
            print(f'  {n + 1}/{len(df)} clips processed')
    X = np.stack(X).astype(np.float32)
    return (X, np.array([r[0] for r in rows]), np.array([r[1] for r in rows]), np.array([r[2] for r in rows]),
            np.array([r[3] for r in rows]), np.array([r[4] for r in rows]))

cfg = [SR, N_FFT, HOP, N_MELS, F_MIN, F_MAX, SILENCE_RMS, REL_LOUD, MAX_WIN_TRAIN, MAX_WIN_EVAL, MAX_SECONDS, SEED]
key = hashlib.md5((df_items[['path', 'label', 'split', 'rand_off']].to_csv(index=False) + json.dumps(cfg)).encode()).hexdigest()[:12]
cache = f'features_{key}.npz'
if os.path.exists(cache):
    print('Loading cached features:', cache)
    z = np.load(cache)
    X, y, split, source, sub, clip = z['X'], z['y'], z['split'], z['source'], z['sub'], z['clip']
else:
    t0 = time.time()
    X, y, split, source, sub, clip = build_features(df_items)
    np.savez(cache, X=X, y=y, split=split, source=source, sub=sub, clip=clip)
    print(f'Features built in {time.time() - t0:.0f}s and cached as {cache}')

rng = np.random.default_rng(SEED)
def cap_indices(idx, cap_class, cap_bg):
    keep = []
    for c in range(NUM_CLASSES):
        ii = idx[y[idx] == c]
        cap = cap_bg if c == BG else cap_class
        if len(ii) > cap:
            ii = rng.choice(ii, cap, replace=False)
        keep.append(ii)
    return np.sort(np.concatenate(keep))

tr_idx = cap_indices(np.where(split == 'train')[0], WIN_CAP_TRAIN_CLASS, WIN_CAP_TRAIN_BG)
va_idx = cap_indices(np.where(split == 'val')[0], 10 ** 9, WIN_CAP_EVAL_BG)
te_idx = cap_indices(np.where(split == 'test')[0], 10 ** 9, WIN_CAP_EVAL_BG)
X_train, y_train = X[tr_idx], y[tr_idx]
X_val, y_val = X[va_idx], y[va_idx]
X_test, y_test = X[te_idx], y[te_idx]
src_test, clip_test = source[te_idx], clip[te_idx]

tbl = pd.DataFrame({'train': np.bincount(y_train, minlength=NUM_CLASSES), 'val': np.bincount(y_val, minlength=NUM_CLASSES),
                    'test': np.bincount(y_test, minlength=NUM_CLASSES)}, index=LABELS)
print('\nWINDOWS per class and split:')
print(tbl.to_string())
all_idx = np.concatenate([tr_idx, va_idx, te_idx])
(pd.DataFrame({'label': [LABELS[i] for i in y[all_idx]], 'source': source[all_idx], 'split': split[all_idx]})
   .groupby(['label', 'source', 'split']).size().rename('windows').reset_index().to_csv('dataset_summary.csv', index=False))
print('\nWINDOWS per class x source (train+val+test) saved to dataset_summary.csv:')
print(pd.crosstab(np.array(LABELS)[y[all_idx]], source[all_idx], margins=True).to_string())

assert (tbl['train'] > 0).all(), 'a class has no TRAIN windows'
for c, row in tbl.iterrows():
    if row['val'] == 0 or row['test'] < 20:
        print(f'  ! {c}: only {row["val"]} val / {row["test"]} test windows -> its scores will be noisy (add more clips)')

FEAT_MEAN, FEAT_STD = float(X_train.mean()), float(X_train.std() + 1e-6)
print(f'\nFeature mean {FEAT_MEAN:.4f} | std {FEAT_STD:.4f}')

# balanced class weights (clipped) so background / big classes don't swamp small ones
cnt = np.bincount(y_train, minlength=NUM_CLASSES)
CW = np.clip(len(y_train) / (NUM_CLASSES * np.maximum(cnt, 1)), 0.25, 4.0).astype(np.float32)

## 5. Model

Same architecture as v3 (raw log-mel in, normalisation inside the model, 128-d L2-normalised embedding for
"Teach a Sound"). The softmax now has **one extra output: background**.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

def conv_block(x, filters, n=2):
    for _ in range(n):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x

inp = layers.Input(shape=(EXPECTED_FRAMES, N_MELS), name='logmel_input')
x = layers.Reshape((EXPECTED_FRAMES, N_MELS, 1))(inp)
x = layers.Rescaling(scale=1.0 / FEAT_STD, offset=-FEAT_MEAN / FEAT_STD, name='normalize')(x)
x = conv_block(x, 32, n=1);  x = layers.MaxPooling2D(2)(x)
x = conv_block(x, 64, n=2);  x = layers.MaxPooling2D(2)(x);  x = layers.Dropout(0.1)(x)
x = conv_block(x, 128, n=2); x = layers.MaxPooling2D(2)(x);  x = layers.Dropout(0.2)(x)
x = conv_block(x, 128, n=1)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu', name='embedding_dense')(x)
x = layers.Dropout(0.3)(x)
embedding = layers.UnitNormalization(axis=-1, name='embedding')(x)
probs_all = layers.Dense(NUM_CLASSES, activation='softmax', name='probs_all')(x)      # trained classes + background

train_model = models.Model(inp, probs_all, name='sonic_train')
print(f'Parameters: {train_model.count_params():,} | outputs: {NUM_CLASSES} ({NUM_TRAINED} app classes + background)')

## 6. Train

Per batch (train only): random **gain** (+-15 dB), time shift, **SpecAugment**, and **power-domain mixup** on half the
samples. Mixing a sound with a background window teaches the model to find the alert inside noise. Label smoothing,
class weights, LR schedule, early stopping on validation loss.

In [ ]:
T, M = EXPECTED_FRAMES, N_MELS

# Online target+background power mixing helps the classifier see events under varied ambient noise.
# The pool contains TRAIN-split background windows only, so no validation/test audio leaks into augmentation.
BG_POOL = tf.constant(X_train[y_train == BG], dtype=tf.float32)
if int(BG_POOL.shape[0]) == 0:
    raise ValueError('No background TRAIN windows available for noise augmentation.')

def augment(x, y, w):
    B = tf.shape(x)[0]
    x = x + tf.random.uniform([B, 1, 1], -GAIN_LOG10, GAIN_LOG10)

    # Small bounded time translation; avoid the full-window circular wrap in v4.
    shift = tf.random.uniform([B], -6, 7, dtype=tf.int32)
    x = tf.map_fn(lambda z: tf.roll(z[0], shift=z[1], axis=0), (x, shift),
                  fn_output_signature=tf.TensorSpec((T, M), tf.float32))

    # Mix a random training background spectrogram into target windows at random SNR.
    bg = tf.gather(BG_POOL, tf.random.uniform([B], 0, tf.shape(BG_POOL)[0], dtype=tf.int32))
    target_mask = tf.argmax(y, axis=1) != BG
    use_mix = (tf.random.uniform([B]) < NOISE_MIX_PROB) & target_mask
    target_power = tf.reduce_mean(tf.pow(10.0, x), axis=[1, 2], keepdims=True)
    bg_power = tf.reduce_mean(tf.pow(10.0, bg), axis=[1, 2], keepdims=True) + 1e-8
    snr_db = tf.random.uniform([B, 1, 1], -5.0, 15.0)
    bg_scale = target_power / bg_power * tf.pow(10.0, -snr_db / 10.0)
    mixed_power = tf.pow(10.0, x) + bg_scale * tf.pow(10.0, bg)
    mixed = tf.math.log(tf.maximum(mixed_power, 1e-6)) / tf.math.log(10.0)
    x = tf.where(use_mix[:, None, None], mixed, x)

    # SpecAugment masks.
    mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
    fw = tf.random.uniform([B, 1, 1], 0, 10, dtype=tf.int32)
    f0 = tf.random.uniform([B, 1, 1], 0, M - 10, dtype=tf.int32)
    tw = tf.random.uniform([B, 1, 1], 0, 12, dtype=tf.int32)
    t0 = tf.random.uniform([B, 1, 1], 0, T - 12, dtype=tf.int32)
    fi, ti = tf.range(M)[None, None, :], tf.range(T)[None, :, None]
    mask = ((fi >= f0) & (fi < f0 + fw)) | ((ti >= t0) & (ti < t0 + tw))
    x = tf.where(mask, mean, x)

    # Conservative power-domain mixup with a random pairing, applied only some of the time.
    perm = tf.random.shuffle(tf.range(B))
    g1, g2 = tf.random.gamma([B], 0.4), tf.random.gamma([B], 0.4)
    lam = g1 / (g1 + g2 + 1e-8)
    lam = tf.maximum(lam, 1.0 - lam)
    lam = tf.where(tf.random.uniform([B]) < MIXUP_PROB, lam, tf.ones([B]))
    l3, l2 = lam[:, None, None], lam[:, None]
    x2, y2, w2 = tf.gather(x, perm), tf.gather(y, perm), tf.gather(w, perm)
    p = l3 * tf.pow(10.0, x) + (1.0 - l3) * tf.pow(10.0, x2)
    x = tf.math.log(tf.maximum(p, 1e-6)) / tf.math.log(10.0)
    y = l2 * y + (1.0 - l2) * y2
    w = lam * w + (1.0 - lam) * w2
    return x, y, w

def onehot(lbl, smooth=0.0):
    return tf.one_hot(lbl, NUM_CLASSES) * (1.0 - smooth) + smooth / NUM_CLASSES

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, onehot(y_train, 0.1), CW[y_train]))
            .shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
            .batch(BATCH, drop_remainder=True).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, onehot(y_val), CW[y_val])).batch(64)

from sklearn.metrics import f1_score

class ValidationMacroF1(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(X_val, batch_size=256, verbose=0).argmax(1)
        score = f1_score(y_val, pred, labels=np.arange(NUM_CLASSES), average='macro', zero_division=0)
        logs['val_macro_f1'] = float(score)
        print(f" - val_macro_f1: {score:.4f}")

train_model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
                    loss=tf.keras.losses.CategoricalCrossentropy(), metrics=['categorical_accuracy'])
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    ValidationMacroF1(),
    tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=PATIENCE,
                                     restore_best_weights=True, verbose=1),
]
history = train_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2)

## 7. Evaluate on the untouched test split

Prints window / smoothed (average of 3 windows) / clip accuracy, per-class precision-recall-F1 (macro-F1 is the
headline, not accuracy), and **accuracy per data source** so a weak source can't hide inside one blended number.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score

p_test = train_model.predict(X_test, batch_size=256, verbose=0)
y_pred = p_test.argmax(1)

clip_p = pd.DataFrame(p_test).groupby(clip_test).mean()
clip_true = pd.Series(y_test).groupby(clip_test).first()
clip_pred = clip_p.values.argmax(1)
clip_acc = accuracy_score(clip_true.loc[clip_p.index], clip_pred)
print(f'Window accuracy (correlated windows; secondary): {accuracy_score(y_test, y_pred):.3f}')
print(f'Clip accuracy (one vote per recording)       : {clip_acc:.3f} ({len(clip_p)} recordings)')
clip_meta = pd.DataFrame({'clip': clip_test, 'source': src_test}).drop_duplicates('clip').set_index('clip')
clip_result = pd.DataFrame({'truth': clip_true, 'pred': clip_p.values.argmax(1)}, index=clip_p.index).join(clip_meta)
print('Clip accuracy by source (realworld_phone is the held-out phone test):')
print(clip_result.assign(ok=clip_result.truth == clip_result.pred).groupby('source').ok.agg(recordings='size', accuracy='mean').round(3).to_string())
print(f'Macro-F1 (all classes) : {f1_score(y_test, y_pred, labels=range(NUM_CLASSES), average="macro", zero_division=0):.3f}\n')
print(classification_report(y_test, y_pred, labels=range(NUM_CLASSES), target_names=LABELS, zero_division=0))

df_t = pd.DataFrame({'source': src_test, 'label': np.array(LABELS)[y_test], 'ok': y_test == y_pred})
print('Accuracy per data source (test; field source is the real-world result):')
print(df_t.groupby('source')['ok'].agg(windows='size', accuracy='mean').round(3).to_string())
print('\nRecall per class x source (test):')
print(df_t.pivot_table(index='label', columns='source', values='ok', aggfunc='mean').round(2).to_string())

fig, ax = plt.subplots(figsize=(11, 11))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=range(NUM_CLASSES), display_labels=LABELS, normalize='true',
                                        xticks_rotation=45, cmap='Blues', values_format='.2f', ax=ax, colorbar=False)
plt.title('Sonic v5: confusion matrix (test, row-normalised)'); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['categorical_accuracy'], label='Train'); axes[0].plot(history.history['val_categorical_accuracy'], label='Val')
axes[0].legend(); axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
axes[1].plot(history.history['loss'], label='Train'); axes[1].plot(history.history['val_loss'], label='Val')
axes[1].legend(); axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

## 8. Pick the alert threshold from validation data

This mirrors what the app does: the app sees only the 20 slots (background is dropped), alerts when the best slot
is above a threshold, and stays silent otherwise. The threshold is chosen on **validation** so that at most
`FA_TARGET` of background windows trigger an alert; the final test split only reports what that choice achieves. Do not tune the threshold using test results.

In [ ]:
def alert_stats(p, y_true, thr):
    pt = p[:, :NUM_TRAINED]                          # the app's view: no background slot
    top, conf = pt.argmax(1), pt.max(1)
    fired = conf >= thr
    is_bg = y_true == BG
    tgt = ~is_bg
    recall = ((fired & (top == y_true)) & tgt).sum() / max(1, tgt.sum())
    wrong = ((fired & (top != y_true)) & tgt).sum() / max(1, tgt.sum())      # alert fired, wrong class
    false_alarm = (fired & is_bg).sum() / max(1, is_bg.sum())
    return recall, false_alarm, wrong

p_val = train_model.predict(X_val, batch_size=256, verbose=0)
thrs = np.round(np.arange(MIN_EVENT_THRESHOLD, 0.951, 0.05), 2)
sweep = pd.DataFrame([(t, *alert_stats(p_val, y_val, t)[:2], *alert_stats(p_test, y_test, t)) for t in thrs],
                     columns=['thr', 'val_recall', 'val_false_alarm', 'test_recall', 'test_false_alarm', 'test_wrong_class'])
print(sweep.round(3).to_string(index=False))
ok = sweep[sweep.val_false_alarm <= FA_TARGET]
THR = float(ok.thr.iloc[0]) if len(ok) else float(thrs[-1])
if not len(ok):
    print(f'\n! No threshold reached a validation false-alarm rate <= {FA_TARGET:.0%}; using {THR}. Add more background data.')
r, fa, wr = alert_stats(p_test, y_test, THR)
print(f'\nRecommended threshold = {THR}  ->  TEST: target-sound recall {r:.3f} | false alarms on background {fa:.3f} | wrong-class alerts {wr:.3f}')

pt = p_test[:, :NUM_TRAINED]
fired_ok = (pt.max(1) >= THR) & (pt.argmax(1) == y_test)
print(f'\nPer-class recall at threshold {THR} (test):')
print(pd.Series({LABELS[c]: fired_ok[y_test == c].mean() for c in range(NUM_TRAINED) if (y_test == c).any()}).round(3).to_string())

## 9. Map into the app's 20 slots + "Teach a Sound" checks

A constant `(N+1) x 20` matrix puts each trained probability into its app slot; the **background row is all zeros**, so
background probability disappears and untrained slots stay 0. Output order: `[class_probs (1x20), embedding (1x128)]`.

Two embedding checks: **seen classes** (optimistic, the model was trained on them) and **never-seen sounds**
(ESC-50 keyboard_typing / mouse_click / clock_tick / can_opening, kept out of training entirely).
That second number is the honest one for the feature, since users teach sounds the model has never heard.

In [ ]:
S = np.zeros((NUM_CLASSES, NUM_APP_CATEGORIES), dtype=np.float32)
for label, slot in enumerate(APP_INDEX):
    S[label, slot] = 1.0                              # row BG stays all zeros
scatter = layers.Dense(NUM_APP_CATEGORIES, use_bias=False, trainable=False, name='class_probs')
class_probs20 = scatter(probs_all)
scatter.set_weights([S])
export_model = models.Model(inp, [class_probs20, embedding], name='sonic_sound_classifier')
embedding_model = models.Model(inp, embedding)

p20, emb = export_model.predict(X_test[:300], verbose=0)
assert np.allclose(p20[:, APP_INDEX], train_model.predict(X_test[:300], verbose=0)[:, :NUM_TRAINED], atol=1e-5)
assert (p20.sum(1) <= 1.0 + 1e-4).all()
untrained = [i for i in range(NUM_APP_CATEGORIES) if i not in APP_INDEX]
if untrained:
    assert np.allclose(p20[:, untrained], 0.0)
bg_mask = y_test[:300] == BG
if bg_mask.any():
    print(f'20-slot mapping verified. Mean top-slot score: background windows {p20[bg_mask].max(1).mean():.2f} '
          f'| target windows {p20[~bg_mask].max(1).mean():.2f}  (lower on background = good)')

from sklearn.metrics import roc_auc_score
def proto_auc(emb_pos_pool, clips_pos, emb_neg_pool, rng_, n_enrol=3):
    ucl = np.unique(clips_pos)
    if len(ucl) < n_enrol + 1:
        return None
    enrol_clips = rng_.choice(ucl, n_enrol, replace=False)
    enrol = np.isin(clips_pos, enrol_clips)
    proto = emb_pos_pool[enrol].mean(0); proto /= np.linalg.norm(proto) + 1e-12
    pos, neg = emb_pos_pool[~enrol] @ proto, emb_neg_pool @ proto
    return roc_auc_score(np.r_[np.ones(len(pos)), np.zeros(len(neg))], np.r_[pos, neg])

rng_probe = np.random.default_rng(0)
emb_test = embedding_model.predict(X_test, batch_size=256, verbose=0)
aucs_seen = {}
for c in range(NUM_TRAINED):
    m = y_test == c
    a = proto_auc(emb_test[m], clip_test[m], emb_test[~m], rng_probe)
    if a is not None:
        aucs_seen[LABELS[c]] = a
print(f'\nEmbedding AUC, SEEN classes (optimistic): {np.mean(list(aucs_seen.values())):.3f}' if aucs_seen else '\nNo seen class had enough test clips for the AUC check.')

aucs_novel = {}
if novel_items:
    nX, ncls, nclip = [], [], []
    prng = np.random.default_rng(1)
    for ci, (path, cname) in enumerate(novel_items):
        try:
            wav = load_wave(path)
        except Exception:
            continue
        for w in pick_windows(wav, 'test', False, prng):
            nX.append(extract_logmel(w)); ncls.append(cname); nclip.append(ci)
    if nX:
        nX, ncls, nclip = np.stack(nX), np.array(ncls), np.array(nclip)
        emb_nov = embedding_model.predict(nX, batch_size=256, verbose=0)
        emb_ref = emb_test[:3000]
        for cname in sorted(set(ncls)):
            m = ncls == cname
            a = proto_auc(emb_nov[m], nclip[m], np.concatenate([emb_nov[~m], emb_ref]), rng_probe)
            if a is not None:
                aucs_novel[cname] = a
if aucs_novel:
    print('Embedding AUC, NEVER-SEEN sounds (the honest number for "Teach a Sound"):')
    print(pd.Series(aucs_novel).round(3).to_string())
    print(f'mean = {np.mean(list(aucs_novel.values())):.3f}   (1.0 perfect, 0.5 useless)')

## 10. Export to TFLite and verify

Same export path as v3: `ExportArchive` with alphabetical dict keys, so output 0 = class probs `[1,20]` and
output 1 = embedding `[1,128]`, which is what the Flutter app expects. Float32 on purpose (quantising can hurt the embedding).
The assertions stop you from deploying a file that disagrees with Keras.

In [ ]:
import keras
shutil.rmtree('sonic_savedmodel', ignore_errors=True)

def serve_fn(x):
    probs20, emb_ = export_model(x)
    return {'a_class_probs': probs20, 'b_embedding': emb_}

archive = keras.export.ExportArchive()
archive.track(export_model)
archive.add_endpoint(name='serving_default', fn=serve_fn,
                     input_signature=[tf.TensorSpec([1, EXPECTED_FRAMES, N_MELS], tf.float32)])
archive.write_out('sonic_savedmodel')
tflite_model = tf.lite.TFLiteConverter.from_saved_model('sonic_savedmodel').convert()
with open('sound_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'Saved sound_classifier.tflite ({len(tflite_model) / 1024:.1f} KB)')

interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp_d, out_d = interp.get_input_details(), interp.get_output_details()
print('Input :', inp_d[0]['shape'], inp_d[0]['dtype'])
for i, o in enumerate(out_d):
    print(f'Output {i}:', o['shape'], o['dtype'])
assert tuple(inp_d[0]['shape']) == (1, EXPECTED_FRAMES, N_MELS), 'input must be [1,61,64]'
assert tuple(out_d[0]['shape']) == (1, NUM_APP_CATEGORIES), 'output 0 must be [1,20]'
assert tuple(out_d[1]['shape']) == (1, 128), 'output 1 must be [1,128]'

sel = np.random.default_rng(3).choice(len(X_test), min(600, len(X_test)), replace=False)
ref_probs, ref_emb = export_model.predict(X_test[sel], verbose=0)
agree = 0; max_diff = max_emb_diff = 0.0; tl_probs = []
for j, i in enumerate(sel):
    interp.set_tensor(inp_d[0]['index'], X_test[i:i + 1]); interp.invoke()
    pr = interp.get_tensor(out_d[0]['index'])[0]; em = interp.get_tensor(out_d[1]['index'])[0]
    tl_probs.append(pr)
    max_diff = max(max_diff, float(np.abs(pr - ref_probs[j]).max()))
    max_emb_diff = max(max_emb_diff, float(np.abs(em - ref_emb[j]).max()))
    agree += int(pr.argmax() == ref_probs[j].argmax())
tl_probs = np.array(tl_probs)
print(f'\nTFLite vs Keras on {len(sel)} test windows: argmax agreement {agree}/{len(sel)} | max prob diff {max_diff:.2e} | max embedding diff {max_emb_diff:.2e}')
assert np.isfinite(max_diff) and max_diff < 1e-3 and np.isfinite(max_emb_diff) and max_emb_diff < 1e-3, 'TFLite output differs from Keras - do NOT deploy this file'
assert agree >= 0.99 * len(sel), 'TFLite predictions disagree with Keras - do NOT deploy this file'

# the app's decision rule, run on the .tflite itself: alert if the best slot >= threshold
ys = y_test[sel]; fired = tl_probs.max(1) >= THR
is_bg = ys == BG
true_slot = np.array([APP_INDEX[c] if c != BG else -1 for c in ys])
print(f'.tflite with threshold {THR}: target-sound recall {((fired & (tl_probs.argmax(1) == true_slot)) & ~is_bg).sum() / max(1, (~is_bg).sum()):.3f} '
      f'| false alarms on background {(fired & is_bg).sum() / max(1, is_bg.sum()):.3f}')
print('Output order verified: 0 = class probs [1,20], 1 = embedding [1,128]  (matches the Flutter app)')

json.dump({
    'categories': APP_CATEGORIES, 'target_classes': TARGET_CATEGORIES,
    'trained': TRAINED, 'trained_slots': APP_INDEX, 'not_trained': NOT_TRAINED,
    'internal_background_reject_class': True, 'minimum_train_clips_per_target': MIN_TRAIN_CLIPS,
    'has_background_class': True, 'background_probability_is_dropped': True,
    'recommended_threshold': THR, 'threshold_chosen_for_val_false_alarm_rate': FA_TARGET,
    'runtime_temporal_agreement': {'window': 3, 'required': 2},
    'feature_mean': FEAT_MEAN, 'feature_std': FEAT_STD,
    'sources_used': found, 'realworld_phone_test_present': bool(found.get('realworld_phone_test', 0)),
    'realworld_test_labels_excluded_untrained': FIELD_TEST_UNTRAINED,
    'realworld_test_non_target_labels_mapped_to_background': field_test_non_target_labels,
    'test_window_accuracy': float(accuracy_score(y_test, p_test.argmax(1))),
    'test_clip_accuracy': float(accuracy_score(clip_true.loc[clip_p.index], clip_p.values.argmax(1))),
    'test_macro_f1': float(f1_score(y_test, p_test.argmax(1), labels=np.arange(NUM_CLASSES), average='macro', zero_division=0)),
    'embedding_auc_seen': aucs_seen, 'embedding_auc_never_seen': aucs_novel,
    'note': ('Normalization is inside the model. Feed raw log10 mel [1,61,64]. The six trained target outputs occupy trained_slots; all other app slots are zero. '
             'The internal background probability is omitted from the 20 app outputs. In Flutter, show a target only when its score passes the validation threshold in two of three overlapping windows; otherwise compare mic dB with a calibrated baseline for Normal/Talking display. '
             'A dB change is not proof of speech and will label other above-baseline sounds as Talking.'),
}, open('model_info.json', 'w'), indent=2)

try:
    from google.colab import files as colab_files
    for f_ in ('sound_classifier.tflite', 'model_info.json', 'dataset_summary.csv'):
        colab_files.download(f_)
except Exception:
    print('(Not on Colab: grab sound_classifier.tflite, model_info.json and dataset_summary.csv from the working folder.)')

## 11. Deploy and report results honestly

1. Review `model_info.json`: `trained` must contain exactly the six targets; `trained_slots` must match the unchanged 20-slot `categories` order. The model also has an internal background/reject output that is intentionally not exported as an app label.
2. The Flutter app must map TFLite indices using `model_info.json` / the fixed `categories` order, not a shorter or differently ordered UI list. Only the six `trained_slots` are eligible to appear as sound labels.
3. Derive one live display state for all result widgets: if the highest of the six target scores meets `recommended_threshold`, show that target; otherwise show `Normal` when the rolling microphone level is within its calibrated baseline, or `Talking` when it departs from baseline. A decibel departure is only an activity proxy, not speech recognition; traffic, music, and other noises above baseline will also show `Talking`. Use a calibrated margin and brief hysteresis to avoid flicker from microphone self-noise.
4. The dB baseline rule belongs in Flutter runtime logic, not in the trained six-class classifier. Calibrate it on the actual phone/microphone and do not treat untrained TFLite slots as detections.
5. Feed raw `log10(max(energy, 1e-6))` values shaped `[1,61,64]`; normalization is inside the model. Average recent probabilities as configured and use the validation-selected threshold (never below 0.70) and require the same class in two of three overlapping windows. Verify Dart and librosa features on the same fixed audio sample.
6. A dataset score is not a real-world guarantee. The supplied environment currently has ESC-50 only (40 recordings per target class); FSD50K, UrbanSound8K, MUSAN, extra recordings, and `realworld_test_manifest.csv` are absent. Add more independent recordings, especially phone-recorded clips, before claiming strong real-world performance. Keep the field test recordings out of training and validation.
